# Data Cleaning

Doing the following data cleaning steps:
- Put dates into datetime format for calculations 
- Remove duplicates
- Remove chances that were ever "interrupted"
- Keep only one entry per day (the last) for each chance (for later variables like amount of changes or volatility)
- Remove backdated entries (anything before 01.02.2019 out because to avoid backfilling bias)
- Remove placeholder deals (deals that are expected to close later than 31.12.2032)
- Drop unnecessary columns: forecastrelevant, chance_topic, pricelist, costcenter_code, org_business_en (mostly because of high NAs, additionally because of no actual meaning for our task)
- Remove rows with NAs for: amount, org_category, stage_en
- Remove chances that were closed within one day (probably backfilled and not useful for predictions)
- Standardize probability (diving by 0.91, as all values above 0.9 are basically "won" already, and then capping maximum value at 1)
- Add the target_is_won variable (Won = 1 / Lost = 0 / Open = NA) to all observations (basically the final state of the chance)

Then, I look at variables I expect to be correlated (status_code, stage_code, days_until_saledate and probability) and the distribution in duration for closed deals, as well as how long open deals have been open for already:
- 61.6% of deals are closed within 90 days (exlcuding the ones closed in 1 day).
- At the same time, 89.3% of open deals are already open for more than 90 days.




In [23]:
import pandas as pd
import numpy as np

# 1. Load data
df = pd.read_csv('../data/chance_changelog.csv')
print(f"Raw dataset: {len(df)} rows, {df['chance_id'].nunique()} chances")

# 2. Core type conversions
for col in ['registered_dt', 'updated_dt', 'saledate']:
    df[col] = pd.to_datetime(df[col], errors='coerce')
df['chance_id'] = df['chance_id'].astype('string')
df['status'] = df['status'].astype('string')

# 3. Remove full-row duplicates
r0, n0 = len(df), df['chance_id'].nunique()
df = df.drop_duplicates(keep='first')
print(f"Full-row duplicates removed:               {r0 - len(df):>6} rows, {n0 - df['chance_id'].nunique():>5} chances")

# 4. Remove chances that ever had status 'Interrupted'
r0, n0 = len(df), df['chance_id'].nunique()
interrupted_ids = df.loc[df['status'] == 'Interrupted', 'chance_id'].unique()
df = df[~df['chance_id'].isin(interrupted_ids)]
print(f"Interrupted chances removed:               {r0 - len(df):>6} rows, {n0 - df['chance_id'].nunique():>5} chances")

# 5. Enforce one entry per day per chance (keep last update per day)
df['updated_date'] = df['updated_dt'].dt.date
df = df.sort_values(['chance_id', 'updated_dt'])
r0, n0 = len(df), df['chance_id'].nunique()
df = df.groupby(['chance_id', 'updated_date'], as_index=False).tail(1)
print(f"One entry per day enforced:                {r0 - len(df):>6} rows, {n0 - df['chance_id'].nunique():>5} chances")

# 6. Remove chances where all observations have probability == 0
r0, n0 = len(df), df['chance_id'].nunique()
zero_prob_ids = df.groupby('chance_id')['probability'].max()
zero_prob_ids = zero_prob_ids[zero_prob_ids == 0].index
df = df[~df['chance_id'].isin(zero_prob_ids)]
print(f"All-zero probability removed:              {r0 - len(df):>6} rows, {n0 - df['chance_id'].nunique():>5} chances")

# 7. Remove intercompany deals
r0, n0 = len(df), df['chance_id'].nunique()
interco_ids = df.loc[df['intercompany_flag'] == 1, 'chance_id'].unique()
df = df[~df['chance_id'].isin(interco_ids)]
print(f"Intercompany deals removed:                {r0 - len(df):>6} rows, {n0 - df['chance_id'].nunique():>5} chances")

# 8. Remove chances with any unrealistic saledate (before 2019-02-01 or after 2032-12-31)
r0, n0 = len(df), df['chance_id'].nunique()
bad_mask = (df['saledate'] < pd.Timestamp('2019-02-01')) | (df['saledate'] > pd.Timestamp('2032-12-31'))
df = df[~df['chance_id'].isin(df.loc[bad_mask, 'chance_id'].unique())]
print(f"Unrealistic saledates removed:             {r0 - len(df):>6} rows, {n0 - df['chance_id'].nunique():>5} chances")

# 9. Drop unused columns
cols_to_drop = ['forecastrelevant', 'chance_topic', 'pricelist', 'costcenter_code', 'org_business_en']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')

# 10. Normalize probability (raw 0-100 -> 0-1, capped at 1)
df['probability'] = (df['probability'] / 100.0) / 0.91
df['probability'] = df['probability'].clip(upper=1.0)

# 11. Remove chances with no usable open observation (status not Won/Lost AND 0 < probability < 1)
#     Runs after normalization so the 0-1 thresholds are correct
r0, n0 = len(df), df['chance_id'].nunique()
open_mask = (~df['status'].isin(['Won', 'Lost'])) & (df['probability'] > 0) & (df['probability'] < 1)
df = df[df['chance_id'].isin(df.loc[open_mask, 'chance_id'].unique())]
print(f"No usable open observation removed:        {r0 - len(df):>6} rows, {n0 - df['chance_id'].nunique():>5} chances")

# 12. Drop rows with missing amount, org_category_en, or stage_en
for col in ['amount', 'org_category_en', 'stage_en']:
    r0, n0 = len(df), df['chance_id'].nunique()
    df = df[df[col].notna()]
    print(f"Missing {col} removed:{(26 - len(col)) * ' '}{r0 - len(df):>6} rows, {n0 - df['chance_id'].nunique():>5} chances")

# 13. Remove deals closed within 1 day
df = df.sort_values(['chance_id', 'updated_dt'])
reg_first = df.groupby('chance_id')['registered_dt'].min()
terminal_first = df.loc[df['status'].isin(['Won', 'Lost'])].groupby('chance_id')['updated_dt'].min()
common_ids = reg_first.index.intersection(terminal_first.index)
deal_length_days = (terminal_first[common_ids] - reg_first[common_ids]).dt.days
fast_close_ids = deal_length_days[deal_length_days <= 1].index
r0, n0 = len(df), df['chance_id'].nunique()
df = df[~df['chance_id'].isin(fast_close_ids)]
print(f"Deals closed within 1 day removed:        {r0 - len(df):>6} rows, {n0 - df['chance_id'].nunique():>5} chances")

print(f"\nAfter cleaning: {len(df)} rows, {df['chance_id'].nunique()} chances")

# 14. Add days_until_saledate and diagnostics
df['days_until_saledate'] = (df['saledate'] - df['updated_dt']).dt.days

# Correlation matrix
corr_df = pd.DataFrame({
    'status_code': df['status'].astype('category').cat.codes,
    'stage_code': df['stage_en'].astype('category').cat.codes,
    'days_until_saledate': df['days_until_saledate'],
    'probability': df['probability'],
})
print("\nCorrelation matrix (status_code, stage_code, days_until_saledate, probability):")
print(corr_df.corr())

# Deal length and open deal age overview (reuses reg_first / terminal_first from step 13)
closed_ids = terminal_first.index
common_ids = reg_first.index.intersection(closed_ids)
deal_length_days = (terminal_first[common_ids] - reg_first[common_ids]).dt.days
total_closed = len(deal_length_days)
print("\nDeal length (days) for closed deals:")
print(deal_length_days.describe(percentiles=[0.5, 0.75, 0.9, 0.95]))
for label, mask in {
    '<=1 day':       deal_length_days <= 1,
    '>1–7 days':    (deal_length_days > 1)  & (deal_length_days <= 7),
    '>7–30 days':   (deal_length_days > 7)  & (deal_length_days <= 30),
    '>30–90 days':  (deal_length_days > 30) & (deal_length_days <= 90),
    '>90 days':      deal_length_days > 90,
}.items():
    count = mask.sum()
    print(f"  {label}: {count} ({count/total_closed*100:.1f}%)")

open_only_ids = reg_first.index.difference(closed_ids)
if len(open_only_ids) > 0:
    reference_date = df['updated_dt'].max()
    age_open_days = (reference_date - reg_first[open_only_ids]).dt.days
    total_open = len(age_open_days)
    print("\nOpen deal ages (days):")
    print(age_open_days.describe(percentiles=[0.5, 0.75, 0.9, 0.95]))
    for label, mask in {
        '<=30 days':      age_open_days <= 30,
        '>30–90 days':   (age_open_days > 30)  & (age_open_days <= 90),
        '>90–365 days':  (age_open_days > 90)  & (age_open_days <= 365),
        '>365 days':      age_open_days > 365,
    }.items():
        count = mask.sum()
        print(f"  {label}: {count} ({count/total_open*100:.1f}%)")

# 15. Finalize: drop helpers, add target variable, save
df = df.drop(columns=['updated_date', 'days_since_last_update'], errors='ignore')
target_map = df.sort_values(['chance_id', 'updated_dt']).groupby('chance_id')['status'].last()
df['target_won'] = df['chance_id'].map(target_map).map({'Won': 1, 'Lost': 0, 'Open': np.nan})

df_clean = df.copy()
df_clean.to_parquet('../data/clean_chance_changelog.parquet', index=False)
print(f"\nSaved clean_chance_changelog.parquet: {len(df_clean)} rows, {df_clean['chance_id'].nunique()} chances, {df_clean.shape[1]} columns")
print(df_clean.columns.tolist())
df_clean.head(5)

Raw dataset: 239815 rows, 38730 chances
Full-row duplicates removed:                 1932 rows,     0 chances
Interrupted chances removed:                 7690 rows,   747 chances
One entry per day enforced:                 79675 rows,     0 chances
All-zero probability removed:                 896 rows,   655 chances
Intercompany deals removed:                  1763 rows,   218 chances
Unrealistic saledates removed:                461 rows,   163 chances
No usable open observation removed:         10627 rows,  6765 chances
Missing amount removed:                        12 rows,     2 chances
Missing org_category_en removed:               38 rows,     6 chances
Missing stage_en removed:                    1220 rows,     0 chances
Deals closed within 1 day removed:          2050 rows,   832 chances

After cleaning: 133451 rows, 29342 chances

Correlation matrix (status_code, stage_code, days_until_saledate, probability):
                     status_code  stage_code  days_until_saledate 

,chance_id,business_unit,responsible,registered_dt,registered_by_id,updated_dt,updated_by_id,status,stage_en,probability,amount,saledate,org_id,org_country,org_category_en,intercompany_flag,chance_type_en,has_products_and_services,days_until_saledate,target_won
173412,00062bd652616a56,1cdd086394501fee,2948227e0c45da98,2024-03-28 10:28:44,2948227e0c45da98,2024-03-28 10:28:47,2948227e0c45da98,Open,Information Stage,0.10989,4386.532748,2024-06-28,8b5f778fd8b4a27f,6b7e1f5b521de19b,Customer,0,General sale,1,91,1.0
173413,00062bd652616a56,1cdd086394501fee,2948227e0c45da98,2024-03-28 10:28:44,2948227e0c45da98,2024-06-10 07:06:50,2948227e0c45da98,Open,Information Stage,0.10989,4386.532748,2024-08-30,8b5f778fd8b4a27f,6b7e1f5b521de19b,Customer,0,General sale,1,80,1.0
173414,00062bd652616a56,1cdd086394501fee,2948227e0c45da98,2024-03-28 10:28:44,2948227e0c45da98,2024-07-05 11:32:41,9a36a705e467ca5d,Won,Information Stage,1.00000,4386.532748,2024-05-15,8b5f778fd8b4a27f,6b7e1f5b521de19b,Customer,0,General sale,1,-52,1.0
117033,00088ac92b37a2d6,6befd145c261113d,63be3517fc2e9f5e,2022-09-02 06:58:36,63be3517fc2e9f5e,2022-09-02 07:09:20,63be3517fc2e9f5e,Open,Proposal stage,0.43956,3973.753019,2022-11-30,a0b8a322be765c84,6b7e1f5b521de19b,Customer,0,General sale,1,88,1.0
117034,00088ac92b37a2d6,6befd145c261113d,63be3517fc2e9f5e,2022-09-02 06:58:36,63be3517fc2e9f5e,2022-09-08 12:55:02,669aae8e6be92b9c,Won,Proposal stage,1.00000,3973.753019,2022-09-07,a0b8a322be765c84,6b7e1f5b521de19b,Customer,0,General sale,1,-2,1.0


# Step 2: Feature Engineering (Within-Chance)

Adding features computed per chance_id with strict "until then" semantics — every row uses only prior observations (updated_dt <= current row), no future information.

**New variables:**
- resp_change: cumulative count of responsible changes
- updates_last_14d, updates_last_30d, updates_last_180d, updates_last_1y: count of observations in last X days
- days_since_last_update: days since previous update
- product_change_flag: 1 if has_products_and_services changed
- close_date_shift: number of times saledate was pushed back
- total_days_pushed: (current saledate - first saledate)
- saledate_delta: (saledate - updated_dt)
- amount_changes: cumulative count of amount changes
- home_country: 1 if org_country is the most frequent one
- deal_age: week number (7 days = 1 week)
- zombie_deal: 1 if no update for 6+ months
- prob_change_last_update: (current probability - previous probability)
- prob_volatility: SD of probability, considering all prior values, 0 for first
- amount_pct_change: (current amount - first amount) / first amount (max value 100)
- update_frequency_trend: updates_last_14d / updates_last_30d


In [24]:
# Load df_clean (or use from previous cell)
try:
    df = df_clean.sort_values(['chance_id', 'updated_dt']).copy()
except NameError:
    df = pd.read_parquet('../data/clean_chance_changelog.parquet')
    df = df.sort_values(['chance_id', 'updated_dt'])
g = df.groupby('chance_id')

# resp_change
df['_is_resp_change'] = (df['responsible'] != g['responsible'].shift(1)) & g['responsible'].shift(1).notna()
df['resp_change'] = g['_is_resp_change'].cumsum().astype(int)

# days_since_last_update
df['prev_updated_dt'] = g['updated_dt'].shift(1)
df['days_since_last_update'] = np.where(
    df['prev_updated_dt'].notna(),
    (df['updated_dt'] - df['prev_updated_dt']).dt.days,
    (df['updated_dt'] - df['registered_dt']).dt.days
)

# product_change_flag
df['_is_product_change'] = (df['has_products_and_services'] != g['has_products_and_services'].shift(1)) & g['has_products_and_services'].shift(1).notna()
df['product_change_flag'] = df['_is_product_change'].astype(int)

# close_date_shift
df['prev_saledate'] = g['saledate'].shift(1)
df['_is_date_push'] = (df['saledate'] > df['prev_saledate']) & df['prev_saledate'].notna()
df['close_date_shift'] = g['_is_date_push'].cumsum().astype(int)

# total_days_pushed
df['saledate_first'] = g['saledate'].transform('first')
df['total_days_pushed'] = (df['saledate'] - df['saledate_first']).dt.days.fillna(0)

# saledate_delta
df['saledate_delta'] = (df['saledate'] - df['updated_dt']).dt.days

# amount_changes
df['_is_amount_change'] = (df['amount'] != g['amount'].shift(1)) & g['amount'].shift(1).notna()
df['amount_changes'] = g['_is_amount_change'].cumsum().astype(int)

# home_country
top_country = df['org_country'].mode()[0]
df['home_country'] = (df['org_country'] == top_country).astype(int)

# deal_age
df['deal_age'] = ((df['updated_dt'] - df['registered_dt']).dt.days / 7).fillna(0).astype(int)

# zombie_deal (12 months = 365 days)
df['zombie_deal'] = (df['days_since_last_update'] >= 365).astype(int)

# updates_last_Xd
def count_updates_in_window(grp, days):
    udt = grp['updated_dt'].values
    out = np.zeros(len(grp), dtype=int)
    for i in range(len(grp)):
        t = pd.Timestamp(udt[i])
        cutoff = t - pd.Timedelta(days=days)
        out[i] = np.sum(udt[:i + 1] >= cutoff)
    return out
for days, col in [(14, 'updates_last_14d'), (30, 'updates_last_30d'), (180, 'updates_last_180d'), (365, 'updates_last_1y')]:
    vals = []
    for _, grp in g:
        vals.extend(count_updates_in_window(grp, days))
    df[col] = vals

# prob_change_last_update: current prob - previous prob (0 for first row)
df['_prob_prev'] = g['probability'].shift(1)
df['prob_change_last_update'] = (df['probability'] - df['_prob_prev']).fillna(0)

# prob_volatility: expanding std of probability up to current row (0 for first row)
df['prob_volatility'] = (
    df.groupby('chance_id')['probability']
    .expanding()
    .std()
    .fillna(0)
    .reset_index(level=0, drop=True)
)

# amount_pct_change: (current amount - first amount) / first amount, capped at 100, 0 if first amount is zero
df['_amount_first'] = g['amount'].transform('first')
df['amount_pct_change'] = np.where(
    df['_amount_first'] != 0,
    (df['amount'] - df['_amount_first']) / df['_amount_first'],
    0.0
)
df['amount_pct_change'] = df['amount_pct_change'].clip(upper=100)

# update_frequency_trend: updates_last_14d / updates_last_30d
# updates_last_30d >= 1 always (current row counted), no division by zero
df['update_frequency_trend'] = df['updates_last_14d'] / df['updates_last_30d']

# Drop helpers
df = df.drop(columns=[
    '_is_resp_change', 'prev_updated_dt', 'prev_saledate', '_is_date_push',
    '_is_amount_change', 'saledate_first', '_is_product_change',
    '_prob_prev', '_amount_first'
], errors='ignore')

df_newfeats = df.copy()
df_newfeats.to_parquet('../data/newfeats_chance_changelog.parquet', index=False)
print(f"Saved newfeats_chance_changelog.parquet: {df_newfeats.shape[0]} rows, {df_newfeats.shape[1]} columns")

def full_overview(df_in, name):
    if len(df_in) == 0:
        print(f"\n{name}: Empty")
        return
    print(f"\n{name}: shape {df_in.shape}")
    rows = []
    for col in df_in.columns:
        s = df_in[col]
        count = s.count()
        n_na = s.isna().sum()
        n_unique = s.nunique()
        if pd.api.types.is_numeric_dtype(s):
            v_min, v_max = s.min(), s.max()
        elif pd.api.types.is_datetime64_any_dtype(s):
            v_min, v_max = s.min(), s.max()
        else:
            v_min, v_max = 'N/A', 'N/A'
        rows.append({'variable': col, 'count': count, 'min': v_min, 'max': v_max, 'unique': n_unique, 'NAs': n_na})
    display(pd.DataFrame(rows).set_index('variable'))

print("\n--- df_newfeats overview ---")
full_overview(df_newfeats, "=== newfeats_chance_changelog ===")
print("\n--- head(5) ---")
display(df_newfeats.head(5))

Saved newfeats_chance_changelog.parquet: 133451 rows, 38 columns

--- df_newfeats overview ---

=== newfeats_chance_changelog ===: shape (133451, 38)


,count,min,max,unique,NAs
variable,,,,,
chance_id,133451,N/A,N/A,29342,0
business_unit,133451,N/A,N/A,7,0
responsible,133451,N/A,N/A,175,0
registered_dt,133451,2019-01-04 06:33:10,2025-12-29 11:40:53,27024,0
registered_by_id,133451,N/A,N/A,193,0
updated_dt,133451,2019-01-04 06:33:58,2026-01-05 16:57:34,133289,0
updated_by_id,133451,N/A,N/A,214,0
status,133451,N/A,N/A,3,0
stage_en,133451,N/A,N/A,10,0



--- head(5) ---


,chance_id,business_unit,responsible,registered_dt,registered_by_id,updated_dt,updated_by_id,status,stage_en,probability,...,deal_age,zombie_deal,updates_last_14d,updates_last_30d,updates_last_180d,updates_last_1y,prob_change_last_update,prob_volatility,amount_pct_change,update_frequency_trend
173412,00062bd652616a56,1cdd086394501fee,2948227e0c45da98,2024-03-28 10:28:44,2948227e0c45da98,2024-03-28 10:28:47,2948227e0c45da98,Open,Information Stage,0.10989,...,0,0,1,1,1,1,0.00000,0.000000,0.0,1.0
173413,00062bd652616a56,1cdd086394501fee,2948227e0c45da98,2024-03-28 10:28:44,2948227e0c45da98,2024-06-10 07:06:50,2948227e0c45da98,Open,Information Stage,0.10989,...,10,0,1,1,2,2,0.00000,0.000000,0.0,1.0
173414,00062bd652616a56,1cdd086394501fee,2948227e0c45da98,2024-03-28 10:28:44,2948227e0c45da98,2024-07-05 11:32:41,9a36a705e467ca5d,Won,Information Stage,1.00000,...,14,0,1,2,3,3,0.89011,0.513905,0.0,0.5
117033,00088ac92b37a2d6,6befd145c261113d,63be3517fc2e9f5e,2022-09-02 06:58:36,63be3517fc2e9f5e,2022-09-02 07:09:20,63be3517fc2e9f5e,Open,Proposal stage,0.43956,...,0,0,1,1,1,1,0.00000,0.000000,0.0,1.0
117034,00088ac92b37a2d6,6befd145c261113d,63be3517fc2e9f5e,2022-09-02 06:58:36,63be3517fc2e9f5e,2022-09-08 12:55:02,669aae8e6be92b9c,Won,Proposal stage,1.00000,...,0,0,2,2,2,2,0.56044,0.396291,0.0,1.0


**Remaining (non-engineered) variables:**

- chance_id (needs to be dropped for the prediction models)
- business_unit (to be included and to be looked at in more detail)
- responsible (to be included)
- registered_dt (not included, replaced by deal_age)
- registered_by_id (to be included for now)
- updated_dt (not included, replaced by days since last update and updates in last xy)
- updated_by_id (to be included for now)
- status (not included, replaced by target_won)
- stage_en (included for now, need to check potential correlation issues with status)
- probability (included for now, need to check potential correlation issues with status)
- amount (to be included)
- saledate (not included, replaced by close date shift, saledate delta and total days pushed)
- org_id (to be included for now)
- org_country (not to be included, replaced by home_country)
- org_category_en (to be included)
- intercompany_flag (to be included)
- chance_type_en (to be included)
- has_products_and_services (to be included)
- days_until_saledate (same as saledate_delta, so should be excluded)
- target_won (target variable for RQ1)

# Step 3: Rep & Company Variables

This is an extra step, as the results should be verified due to higher complexity. We are adding rep-level features computed with a chronological pass over all rows (sorted by updated_dt). Maintain state: chance_id -> {responsible, amount, status}. For each row, update state, then compute:

- rep_amount: Count of chance_ids where responsible == current row's responsible (as of current updated_dt)
- deal_share_rep: This deal's amount / sum of all deals for this rep (share as ratio 0–1)
- rep_closing_rate: won / (won + lost) for this rep among closed deals (as of current updated_dt)
- customer_win_ratio: amount of deals won with this customer / closed deals with this customer until now 
- is_repeat_customer: 1 if customer was won before

In [25]:
# Rep & Customer Variables: chronological pass over all rows
try:
    df = df_newfeats.sort_values('updated_dt').copy()
except NameError:
    df = pd.read_parquet('data/newfeats_chance_changelog.parquet')
    df = df.sort_values('updated_dt')

state = {}  # chance_id -> {responsible, amount, status, org_id}

# Rep variable lists
rep_amount_list = []
deal_share_rep_list = []
rep_closing_rate_list = []

# Customer variable lists
customer_win_ratio_list = []
is_repeat_customer_list = []

# Running customer counters (O(n) — avoids iterating state for each row)
customer_won_counts = {}    # org_id -> cumulative won count
customer_closed_counts = {} # org_id -> cumulative closed count

for idx, row in df.iterrows():
    cid = row['chance_id']
    rep = row['responsible']
    amt = row['amount']
    st = row['status']
    org_id = row['org_id']

    # --- Customer counters: update based on status transition ---
    old = state.get(cid, {})
    old_status = old.get('status', None)
    old_org = old.get('org_id', None)

    if old_status != st or old_org != org_id:
        # Remove old contribution if it was a closed deal
        if old_status in ['Won', 'Lost'] and old_org is not None:
            customer_closed_counts[old_org] = customer_closed_counts.get(old_org, 0) - 1
            if old_status == 'Won':
                customer_won_counts[old_org] = customer_won_counts.get(old_org, 0) - 1
        # Add new contribution if now closed
        if st in ['Won', 'Lost']:
            customer_closed_counts[org_id] = customer_closed_counts.get(org_id, 0) + 1
            if st == 'Won':
                customer_won_counts[org_id] = customer_won_counts.get(org_id, 0) + 1

    # --- Update state ---
    state[cid] = {'responsible': rep, 'amount': amt, 'status': st, 'org_id': org_id}

    # --- Rep variables ---
    rep_chances = [(c, s) for c, s in state.items() if s['responsible'] == rep]
    rep_amount_list.append(len(rep_chances))

    rep_total = sum(s['amount'] for _, s in rep_chances)
    deal_share = amt / rep_total if rep_total > 0 else np.nan
    deal_share_rep_list.append(deal_share)

    rep_closed = [s for _, s in rep_chances if s['status'] in ['Won', 'Lost']]
    won_rep = sum(1 for s in rep_closed if s['status'] == 'Won')
    lost_rep = sum(1 for s in rep_closed if s['status'] == 'Lost')
    rate = won_rep / (won_rep + lost_rep) if (won_rep + lost_rep) > 0 else 0.0
    rep_closing_rate_list.append(rate)

    # --- Customer variables ---
    closed = customer_closed_counts.get(org_id, 0)
    won_cust = customer_won_counts.get(org_id, 0)
    customer_win_ratio_list.append(won_cust / closed if closed > 0 else 0.0)
    is_repeat_customer_list.append(1 if won_cust > 0 else 0)

df['rep_amount'] = rep_amount_list
df['deal_share_rep'] = deal_share_rep_list
df['rep_closing_rate'] = rep_closing_rate_list
df['customer_win_ratio'] = customer_win_ratio_list
df['is_repeat_customer'] = is_repeat_customer_list

# Sanity checks
max_rep = df['rep_amount'].max()
reps_with_max = df.loc[df['rep_amount'] == max_rep, 'responsible'].unique().tolist()
print(f"rep_amount: max={max_rep}, reps with max: {reps_with_max[:3]}{'...' if len(reps_with_max) > 3 else ''}")
print(f"rep_amount percentiles: {df['rep_amount'].quantile([0.5, 0.9, 0.95, 0.99]).to_dict()}")
print(f"customer_win_ratio: min={df['customer_win_ratio'].min():.3f}, max={df['customer_win_ratio'].max():.3f}, mean={df['customer_win_ratio'].mean():.3f}")
print(f"is_repeat_customer: {df['is_repeat_customer'].value_counts().to_dict()}")

df_repvars = df.copy()
df_repvars.to_parquet('../data/newfeats_repvars_chance_changelog.parquet', index=False)
print(f"Saved newfeats_repvars_chance_changelog.parquet: {df_repvars.shape[0]} rows, {df_repvars.shape[1]} columns")

print("\n--- df_repvars overview ---")
full_overview(df_repvars, "=== newfeats_repvars_chance_changelog ===")
print("\n--- head(5) ---")
display(df_repvars.head(5))

rep_amount: max=1445, reps with max: ['63be3517fc2e9f5e']
rep_amount percentiles: {0.5: 249.0, 0.9: 850.0, 0.95: 996.0, 0.99: 1243.0}
customer_win_ratio: min=0.000, max=1.000, mean=0.531
is_repeat_customer: {1: 102570, 0: 30881}
Saved newfeats_repvars_chance_changelog.parquet: 133451 rows, 43 columns

--- df_repvars overview ---

=== newfeats_repvars_chance_changelog ===: shape (133451, 43)


,count,min,max,unique,NAs
variable,,,,,
chance_id,133451,N/A,N/A,29342,0
business_unit,133451,N/A,N/A,7,0
responsible,133451,N/A,N/A,175,0
registered_dt,133451,2019-01-04 06:33:10,2025-12-29 11:40:53,27024,0
registered_by_id,133451,N/A,N/A,193,0
updated_dt,133451,2019-01-04 06:33:58,2026-01-05 16:57:34,133289,0
updated_by_id,133451,N/A,N/A,214,0
status,133451,N/A,N/A,3,0
stage_en,133451,N/A,N/A,10,0



--- head(5) ---


,chance_id,business_unit,responsible,registered_dt,registered_by_id,updated_dt,updated_by_id,status,stage_en,probability,...,updates_last_1y,prob_change_last_update,prob_volatility,amount_pct_change,update_frequency_trend,rep_amount,deal_share_rep,rep_closing_rate,customer_win_ratio,is_repeat_customer
28,87f0ee439fdb2c4f,ac2a5ee6878db2c2,f455ed364d29b6d0,2019-01-04 06:33:10,f455ed364d29b6d0,2019-01-04 06:33:58,f455ed364d29b6d0,Open,Proposal stage,0.329670,...,1,0.0,0.0,0.0,1.0,1,1.000000,0.0,0.0,0
31,6c447777ff78a0d9,ac2a5ee6878db2c2,f455ed364d29b6d0,2019-01-04 07:37:42,f455ed364d29b6d0,2019-01-04 07:45:43,f455ed364d29b6d0,Open,Proposal stage,0.439560,...,1,0.0,0.0,0.0,1.0,2,0.894161,0.0,0.0,0
42,97136ba7230740e9,6befd145c261113d,acc2ad60b295c7d8,2019-01-04 09:49:18,acc2ad60b295c7d8,2019-01-04 10:52:25,f69ff79d25524c88,Open,Proposal stage,0.989011,...,1,0.0,0.0,0.0,1.0,1,1.000000,0.0,0.0,0
78,8caf86b0bfe0e84d,6befd145c261113d,acc2ad60b295c7d8,2019-01-04 11:36:48,acc2ad60b295c7d8,2019-01-04 11:37:05,acc2ad60b295c7d8,Open,Proposal stage,0.769231,...,1,0.0,0.0,0.0,1.0,2,0.853195,0.0,0.0,0
102,58e44297b976f63d,4928d3c96935fb3f,13e9b974b083387c,2019-01-04 13:34:24,32c34f577904e76d,2019-01-04 14:05:41,32c34f577904e76d,Open,Proposal stage,0.109890,...,1,0.0,0.0,0.0,1.0,1,1.000000,0.0,0.0,0


# Step 4: Entries per Chance Analysis

Compute distribution of rows per chance_id. -> from this we learn that creating a standardized df with 10 observations seems to be the best choice.

In [26]:
# Entries per chance
entries_per_chance = df_repvars.groupby('chance_id').size()
stats = entries_per_chance.describe(percentiles=[0.05, 0.95])
print("Entries per chance:")
print(f"  min: {entries_per_chance.min()}")
print(f"  max: {entries_per_chance.max()}")
print(f"  5th percentile: {entries_per_chance.quantile(0.05)}")
print(f"  10th percentile: {entries_per_chance.quantile(0.10)}")
print(f"  10th percentile: {entries_per_chance.quantile(0.50)}")
print(f"  90th percentile: {entries_per_chance.quantile(0.90)}")
print(f"  95th percentile: {entries_per_chance.quantile(0.95)}")
print("\nDescribe:")
print(entries_per_chance.describe())


Entries per chance:
  min: 1
  max: 95
  5th percentile: 1.0
  10th percentile: 1.0
  10th percentile: 3.0
  90th percentile: 9.0
  95th percentile: 12.0

Describe:
count    29342.000000
mean         4.548122
std          4.224784
min          1.000000
25%          2.000000
50%          3.000000
75%          6.000000
max         95.000000
dtype: float64


# Chance level table (is it possible to put this elsewhere?)

One row per chance_id, with basic fields and duration metrics

In [5]:
df_sorted.columns

Index(['chance_id', 'business_unit', 'responsible', 'registered_dt',
       'registered_by_id', 'updated_dt', 'updated_by_id', 'status', 'stage_en',
       'probability', 'amount', 'saledate', 'org_id', 'org_country',
       'org_category_en', 'intercompany_flag', 'chance_type_en',
       'has_products_and_services', 'updated_date', 'days_until_saledate'],
      dtype='str')

In [6]:
# Ensure time columns are proper datetimes
df_sorted = df_sorted.copy()
df_sorted["registered_dt"] = pd.to_datetime(df_sorted["registered_dt"])
df_sorted["updated_dt"] = pd.to_datetime(df_sorted["updated_dt"])
df_sorted["saledate"] = pd.to_datetime(df_sorted["saledate"], errors="coerce")

# Sort by time
df_sorted = df_sorted.sort_values("updated_dt")

# First and last update per chance
agg_first_last = (
    df_sorted.groupby("chance_id")["updated_dt"]
    .agg(first_update="min", last_update="max")
)

# Final status per chance == status at last update
final_status = (
    df_sorted.sort_values("updated_dt")
    .groupby("chance_id")["status"]
    .last()
    .rename("final_status")
)

# Close date per chance
# If 'saledate' is the final close date, we can take the last non-null saledate.
# If status Won/Lost is what defines closing, this still works as long as saledate is set.
close_date = (
    df_sorted
    .loc[df_sorted["saledate"].notna()]
    .sort_values("updated_dt")
    .groupby("chance_id")["saledate"]
    .last()
    .rename("close_dt")
)

# Static info per chance (first time it was registered)
static_cols = [
    "business_unit",
    "responsible",
    "registered_dt",
    "amount",
    "org_country",
    "org_category_en",
    "intercompany_flag",
    "chance_type_en",
    "has_products_and_services",
]
static_info = (
    df_sorted.sort_values("registered_dt")
    .groupby("chance_id")[static_cols]
    .first()
)

# Combine everything into opportunity-level table
df_opp = (
    static_info
    .join([agg_first_last, final_status, close_date])
    .reset_index()
)

# Deal length for closed deals (registration -> close)
df_opp["deal_length_days"] = (
    df_opp["close_dt"] - df_opp["registered_dt"]
).dt.days

# Age of deals (for open ones) relative to latest update in the data
reference_date = df_sorted["updated_dt"].max()
df_opp["age_open_days"] = (
    reference_date - df_opp["registered_dt"]
).dt.days

# Flags
df_opp["is_closed"] = df_opp["final_status"].isin(["Won", "Lost"])
df_opp["is_won"] = df_opp["final_status"].eq("Won")

# Fast closure: closed and very short deal length
df_opp["is_fast_closure"] = df_opp["is_closed"] & (df_opp["deal_length_days"] <= 1)

df_opp.head()


,chance_id,business_unit,responsible,registered_dt,amount,org_country,org_category_en,intercompany_flag,chance_type_en,has_products_and_services,first_update,last_update,final_status,close_dt,deal_length_days,age_open_days,is_closed,is_won,is_fast_closure
0,00062bd652616a56,1cdd086394501fee,2948227e0c45da98,2024-03-28 10:28:44,4386.532748,6b7e1f5b521de19b,Customer,0,General sale,1,2024-03-28 10:28:47,2024-07-05 11:32:41,Won,2024-05-15,47,648,True,True,False
1,00088ac92b37a2d6,6befd145c261113d,63be3517fc2e9f5e,2022-09-02 06:58:36,3973.753019,6b7e1f5b521de19b,Customer,0,General sale,1,2022-09-02 07:09:20,2022-09-08 12:55:02,Won,2022-09-07,4,1221,True,True,False
2,000a3844cd688374,6befd145c261113d,48084acd5c7780e3,2022-09-28 09:52:06,811.186597,6b7e1f5b521de19b,Customer,0,General sale,1,2022-09-28 16:01:29,2022-10-06 08:31:12,Won,2022-10-06,7,1195,True,True,False
3,000bc1f6564f08df,6befd145c261113d,0df50dd4722ef131,2022-04-01 11:15:07,22962.314002,6b7e1f5b521de19b,Customer,0,General sale,1,2022-04-01 11:15:08,2022-04-11 05:32:59,Won,2022-04-11,9,1375,True,True,False
4,000cfeeeeecd559a,1cdd086394501fee,f324bd92890b7814,2025-01-15 13:38:15,7622.275920,a0d28553c7be77f0,Customer,0,General sale,0,2025-01-15 13:38:16,2025-09-29 06:03:56,Won,2025-09-29,256,355,True,True,False


In [7]:
# Event-level table: one row per update event
events = df_sorted[
    [
        "chance_id",
        "responsible",
        "updated_dt",
        "probability",
        "saledate",
        "status",
        "amount",
    ]
].copy()

# Ensure proper dtypes
events["updated_dt"] = pd.to_datetime(events["updated_dt"])
events["saledate"] = pd.to_datetime(events["saledate"], errors="coerce")

# Sort within each chance by time
events = events.sort_values(["chance_id", "updated_dt"]).reset_index(drop=True)

# Lagged values within each chance to compute changes
events["prob_prev"] = events.groupby("chance_id")["probability"].shift(1)
events["saledate_prev"] = events.groupby("chance_id")["saledate"].shift(1)

# Probability changes
events["prob_change"] = events["probability"] - events["prob_prev"]
events["abs_prob_change"] = events["prob_change"].abs()

# Close-date shifts (in days)
events["saledate_shift_days"] = (
    events["saledate"] - events["saledate_prev"]
).dt.days

# Aggregate behavior metrics per chance
behav_agg = events.groupby("chance_id").agg(
    num_updates=("updated_dt", "count"),
    first_update_event=("updated_dt", "min"),
    last_update_event=("updated_dt", "max"),
    prob_volatility=("probability", "std"),
    mean_abs_prob_change=("abs_prob_change", "mean"),
    num_saledate_shifts=("saledate_shift_days", lambda x: x.notna().sum()),
    total_saledate_shift_days=(
        "saledate_shift_days",
        lambda x: x.fillna(0).abs().sum(),
    ),
)

# Update frequency per month
duration_days = (behav_agg["last_update_event"] - behav_agg["first_update_event"]).dt.days
behav_agg["update_freq_per_month"] = behav_agg["num_updates"] / (duration_days / 30.0).replace(0, np.nan)

behav_agg.head()


,num_updates,first_update_event,last_update_event,prob_volatility,mean_abs_prob_change,num_saledate_shifts,total_saledate_shift_days,update_freq_per_month
chance_id,,,,,,,,
00062bd652616a56,3,2024-03-28 10:28:47,2024-07-05 11:32:41,0.513905,0.445055,2,170.0,0.909091
00088ac92b37a2d6,2,2022-09-02 07:09:20,2022-09-08 12:55:02,0.396291,0.560440,1,84.0,10.000000
000a3844cd688374,3,2022-09-28 16:01:29,2022-10-06 08:31:12,0.513905,0.445055,2,83.0,12.857143
000bc1f6564f08df,3,2022-04-01 11:15:08,2022-04-11 05:32:59,0.513905,0.445055,2,57.0,10.000000
000cfeeeeecd559a,2,2025-01-15 13:38:16,2025-09-29 06:03:56,0.629403,0.890110,1,16.0,0.234375


In [9]:
# Join behavior metrics onto opportunity-level table
df_features = (
    df_opp
    .set_index("chance_id")
    .join(behav_agg)
    .reset_index()
)

df_features.head()


,chance_id,business_unit,responsible,registered_dt,amount,org_country,org_category_en,intercompany_flag,chance_type_en,has_products_and_services,...,is_won,is_fast_closure,num_updates,first_update_event,last_update_event,prob_volatility,mean_abs_prob_change,num_saledate_shifts,total_saledate_shift_days,update_freq_per_month
0,00062bd652616a56,1cdd086394501fee,2948227e0c45da98,2024-03-28 10:28:44,4386.532748,6b7e1f5b521de19b,Customer,0,General sale,1,...,True,False,3,2024-03-28 10:28:47,2024-07-05 11:32:41,0.513905,0.445055,2,170.0,0.909091
1,00088ac92b37a2d6,6befd145c261113d,63be3517fc2e9f5e,2022-09-02 06:58:36,3973.753019,6b7e1f5b521de19b,Customer,0,General sale,1,...,True,False,2,2022-09-02 07:09:20,2022-09-08 12:55:02,0.396291,0.560440,1,84.0,10.000000
2,000a3844cd688374,6befd145c261113d,48084acd5c7780e3,2022-09-28 09:52:06,811.186597,6b7e1f5b521de19b,Customer,0,General sale,1,...,True,False,3,2022-09-28 16:01:29,2022-10-06 08:31:12,0.513905,0.445055,2,83.0,12.857143
3,000bc1f6564f08df,6befd145c261113d,0df50dd4722ef131,2022-04-01 11:15:07,22962.314002,6b7e1f5b521de19b,Customer,0,General sale,1,...,True,False,3,2022-04-01 11:15:08,2022-04-11 05:32:59,0.513905,0.445055,2,57.0,10.000000
4,000cfeeeeecd559a,1cdd086394501fee,f324bd92890b7814,2025-01-15 13:38:15,7622.275920,a0d28553c7be77f0,Customer,0,General sale,0,...,True,False,2,2025-01-15 13:38:16,2025-09-29 06:03:56,0.629403,0.890110,1,16.0,0.234375


# Model Training Dataset Preparation

Build a model-ready dataset with 10 evenly-spaced snapshots per closed chance:
1. Filter to closed chances only (Won/Lost)
2. Compute first closure date per chance
3. Create 10 timestamps at 1/11, 2/11, ..., 10/11 of the open period
4. Filter changelog to open rows (status not Won/Lost, probability in (0,1)), then merge_asof — exactly 10 rows per chance
5. Split by registered_dt: train = before 2025, test = 2025+
6. Drop: chance_id, registered_dt, updated_dt, status, saledate, org_country, days_until_saledate
7. Label-encode categorical variables for modeling

In [29]:
# Model Training Dataset Preparation
# Step 1: Filter to closed chances only
df_closed = df_repvars[df_repvars['target_won'].notna()].copy()
closed_ids = df_closed['chance_id'].unique()
print(f"Closed chances: {len(closed_ids)}")

# Step 2: Compute first closure date per chance
closed_sorted = df_closed.sort_values(['chance_id', 'updated_dt'])
closure_mask = (closed_sorted['status'].isin(['Won', 'Lost'])) | (closed_sorted['probability'] >= 1)
first_closure = closed_sorted[closure_mask].groupby('chance_id')['updated_dt'].min()

# Step 2.5: Filter changelog to open rows (needed for first_open_dt)
changelog = df_closed.sort_values(['chance_id', 'updated_dt'])
open_mask = (~changelog['status'].isin(['Won', 'Lost'])) & (changelog['probability'] > 0) & (changelog['probability'] < 1)
changelog_open = changelog[open_mask]
first_open = changelog_open.groupby('chance_id')['updated_dt'].min()

# Step 3: Create 10 evenly-spaced timestamps per chance
SPLIT_DATE = pd.Timestamp('2025-01-01')
timestamp_rows = []
skipped_neg = []   # chances where open_days <= 0
skipped_empty = [] # chances where no open rows found in merge

closed_ids = [c for c in closed_ids if c in first_closure.index and c in first_open.index]
for cid in closed_ids:
    start = first_open.loc[cid]
    end = first_closure.loc[cid]
    open_days = (end - start).total_seconds() / 86400
    if open_days <= 0:
        # Fix: use a 1-day minimum window anchored at first_closure - 1 day
        start = end - pd.Timedelta(days=1)
        open_days = 1.0
        skipped_neg.append(cid)
    for i in range(1, 11):
        frac = i / 11
        ts = start + pd.Timedelta(days=open_days * frac)
        timestamp_rows.append({'chance_id': cid, 'artificial_ts': ts})
df_timestamps = pd.DataFrame(timestamp_rows)

print(f"Chances fixed (open_days <= 0): {len(skipped_neg)}")

# Step 4: merge_asof
merged_parts = []
for cid in df_timestamps['chance_id'].unique():
    left = df_timestamps[df_timestamps['chance_id'] == cid].sort_values('artificial_ts')
    right = changelog_open[changelog_open['chance_id'] == cid].drop(columns=['chance_id'], errors='ignore')
    if len(left) == 0 or len(right) == 0:
        skipped_empty.append(cid)
        continue
    m = pd.merge_asof(left, right, left_on='artificial_ts', right_on='updated_dt', direction='backward')
    # For fixed chances (open_days was <= 0), merge_asof direction='backward' may not find a match
    # since all open rows are after the artificial timestamps — fall back to 'forward' for those
    if m['updated_dt'].isna().all() and cid in skipped_neg:
        m = pd.merge_asof(left, right, left_on='artificial_ts', right_on='updated_dt', direction='forward')
    merged_parts.append(m)

print(f"Chances skipped (no open rows for merge): {len(skipped_empty)}")

df_model = pd.concat(merged_parts, ignore_index=True)
rows_per_chance = df_model.groupby('chance_id').size()
print(f"Model dataset: {len(df_model)} rows across {rows_per_chance.index.nunique()} chances")
print(f"Rows per chance — min: {rows_per_chance.min()}, max: {rows_per_chance.max()}, "
      f"exactly 10: {(rows_per_chance == 10).sum()}")

# Add registered_dt for split (from full changelog)
reg_map = changelog.groupby('chance_id')['registered_dt'].first()
df_model['registered_dt'] = df_model['chance_id'].map(reg_map)

# Step 5: Train/test split by registered_dt
train_mask = df_model['registered_dt'] < SPLIT_DATE
test_mask = df_model['registered_dt'] >= SPLIT_DATE
df_train_raw = df_model[train_mask].copy()
df_test_raw = df_model[test_mask].copy()
print(f"Train: {len(df_train_raw)} rows, Test: {len(df_test_raw)} rows")

# Step 6: Drop columns
cols_drop = ['chance_id', 'registered_dt', 'updated_dt', 'status', 'saledate', 'org_country', 'days_until_saledate', 'artificial_ts']
cols_drop = [c for c in cols_drop if c in df_train_raw.columns]
df_train = df_train_raw.drop(columns=cols_drop, errors='ignore')
df_test = df_test_raw.drop(columns=cols_drop, errors='ignore')

# Step 7: Label-encode categoricals
from sklearn.preprocessing import LabelEncoder
cat_cols = ['business_unit', 'responsible', 'registered_by_id', 'updated_by_id', 'stage_en', 'org_id', 'org_category_en', 'chance_type_en']
for col in cat_cols:
    if col in df_train.columns:
        le = LabelEncoder()
        combined = pd.concat([df_train[col].astype(str), df_test[col].astype(str)], ignore_index=True)
        le.fit(combined)
        df_train[col] = le.transform(df_train[col].astype(str))
        df_test[col] = le.transform(df_test[col].astype(str))
print(f"Final df_train: {df_train.shape}, df_test: {df_test.shape}")
print(f"Target balance train: Won={df_train['target_won'].sum():.0f}, Lost={(df_train['target_won'] == 0).sum():.0f}")

df_train.to_parquet('../data/model_train.parquet', index=False)
df_test.to_parquet('../data/model_test.parquet', index=False)
print(f"Saved model_train.parquet ({len(df_train)} rows) and model_test.parquet ({len(df_test)} rows) to ../data/")

Closed chances: 23850
Chances fixed (open_days <= 0): 41
Chances skipped (no open rows for merge): 0
Model dataset: 238500 rows across 23850 chances
Rows per chance — min: 10, max: 10, exactly 10: 23850
Train: 209140 rows, Test: 29360 rows
Final df_train: (209140, 36), df_test: (29360, 36)
Target balance train: Won=118100, Lost=91040
Saved model_train.parquet (209140 rows) and model_test.parquet (29360 rows) to ../data/


In [30]:
# Quick check: closed chances with no "open" rows (status not Won/Lost, 0 < prob < 1)
closed_ids = df_repvars[df_repvars['target_won'].notna()]['chance_id'].unique()
open_mask = (~df_repvars['status'].isin(['Won', 'Lost'])) & (df_repvars['probability'] > 0) & (df_repvars['probability'] < 1)
chances_with_open = df_repvars[open_mask]['chance_id'].unique()
chances_without_open = set(closed_ids) - set(chances_with_open)
print(f"Closed chances: {len(closed_ids)}, with at least one open row: {len(chances_with_open)}, with no open row: {len(chances_without_open)}")

# NA check for model_train and model_test
print("\n--- NA check (model_train, model_test) ---")
print(f"model_train: {df_train.isna().sum().sum()} NAs total")
print(f"model_test: {df_test.isna().sum().sum()} NAs total")
na_train = df_train.isna().sum()
na_test = df_test.isna().sum()
if na_train.sum() > 0 or na_test.sum() > 0:
    print("Columns with NAs:")
    for col in df_train.columns:
        if na_train[col] > 0 or na_test[col] > 0:
            print(f"  {col}: train={na_train[col]}, test={na_test[col]}")

Closed chances: 23850, with at least one open row: 29342, with no open row: 0

--- NA check (model_train, model_test) ---
model_train: 0 NAs total
model_test: 0 NAs total
